<a href="https://colab.research.google.com/github/karquiza/bikeshare-toronto-demand/blob/main/notebooks/03_testing_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Working code

"""
Match Toronto Bike Share stations to the nearest Environment Canada (ECCC)
weather station and merge daily temperature / precipitation data.

pip install pandas numpy requests openpyxl
"""
import io
import time
import numpy as np
import pandas as pd
import requests

# ---------------- settings ----------------
START = "2024-01-01"
END = "2024-1-31"
PROVINCE = "ONTARIO"          # restrict candidates to this province
MAX_KM = 25                   # ignore weather stations farther than this
MIN_COVERAGE_YEARS = True     # require station to report daily data for the range
OUT_XLSX = "bike_stations_weather.xlsx"
OUT_CSV = "bike_stations_weather.csv"
# ------------------------------------------

BIKE_URL = "https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
INVENTORY_URL = ("https://collaboration.cmc.ec.gc.ca/cmc/climate/"
                 "Get_More_Data_Plus_de_donnees/Station%20Inventory%20EN.csv")
BULK_URL = "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"

start, end = pd.Timestamp(START), pd.Timestamp(END)


# 1. Bike stations -----------------------------------------------------------
def get_bike_stations():
    data = requests.get(BIKE_URL, timeout=30).json()["data"]["stations"]
    df = pd.DataFrame(data)
    keep = [c for c in ["station_id", "short_name", "name", "lat", "lon"] if c in df]
    return df[keep].rename(columns={"lat": "bike_lat", "lon": "bike_lon"})


# 2. ECCC station inventory --------------------------------------------------
def get_weather_stations():
    r = requests.get(INVENTORY_URL, timeout=60)
    r.raise_for_status()
    inv = pd.read_csv(io.StringIO(r.content.decode("utf-8-sig")), skiprows=3)
    inv = inv.rename(columns={
        "Latitude (Decimal Degrees)": "wx_lat",
        "Longitude (Decimal Degrees)": "wx_lon",
        "DLY First Year": "dly_first",
        "DLY Last Year": "dly_last",
    })
    inv = inv[inv["Province"].str.upper() == PROVINCE]
    inv = inv.dropna(subset=["wx_lat", "wx_lon", "dly_first", "dly_last"])
    if MIN_COVERAGE_YEARS:
        inv = inv[(inv["dly_first"] <= start.year) & (inv["dly_last"] >= end.year)]
    return inv[["Name", "Station ID", "wx_lat", "wx_lon"]].reset_index(drop=True)


# 3. Nearest weather station & neighbouring bike station calculations --------
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def find_nearest_bike_station(bikes):
    """Calculates the closest neighboring bike station, avoiding self-matching."""
    blat = bikes["bike_lat"].to_numpy()
    blon = bikes["bike_lon"].to_numpy()
    names = bikes["name"].to_numpy()

    near_names = []
    near_dists = []

    for i, (lat, lon) in enumerate(zip(blat, blon)):
        dists = haversine_km(lat, lon, blat, blon)
        dists[i] = np.inf  # Prevent station from matching with itself
        nearest_idx = dists.argmin()
        near_names.append(names[nearest_idx])
        near_dists.append(dists[nearest_idx])

    out = bikes.copy()
    out["nearest_bike_station_name"] = near_names
    out["nearest_bike_station_dist_km"] = np.round(near_dists, 2)
    return out


def match_stations(bikes, wx):
    wl, wo = wx["wx_lat"].to_numpy(), wx["wx_lon"].to_numpy()
    idx, dist = [], []
    for lat, lon in zip(bikes["bike_lat"], bikes["bike_lon"]):
        d = haversine_km(lat, lon, wl, wo)
        i = int(d.argmin())
        idx.append(i)
        dist.append(d[i])
    out = bikes.copy()
    out["wx_station_id"] = wx.loc[idx, "Station ID"].to_numpy()
    out["wx_name"] = wx.loc[idx, "Name"].to_numpy()  # Shows nearest weather station name
    out["wx_lat"] = wl[idx]
    out["wx_lon"] = wo[idx]
    out["dist_km"] = np.round(dist, 2)
    return out[out["dist_km"] <= MAX_KM]


# 4. Download daily data (one request per year per station) ------------------
def download_daily(station_id):
    frames = []
    for year in range(start.year, end.year + 1):
        params = dict(format="csv", stationID=int(station_id), Year=year,
                      Month=1, Day=1, timeframe=2, submit="Download+Data")
        for attempt in range(3):
            try:
                r = requests.get(BULK_URL, params=params, timeout=60)
                r.raise_for_status()
                break
            except requests.RequestException:
                time.sleep(2 * (attempt + 1))
        else:
            print(f"  failed: station {station_id} year {year}")
            continue
        df = pd.read_csv(io.StringIO(r.content.decode("utf-8-sig")))
        frames.append(df)
        time.sleep(0.5)  # be polite to the server
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["Date/Time"])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    cols = {"Max Temp (°C)": "max_temp_c", "Min Temp (°C)": "min_temp_c",
            "Mean Temp (°C)": "mean_temp_c", "Total Rain (mm)": "rain_mm",
            "Total Snow (cm)": "snow_cm", "Total Precip (mm)": "precip_mm"}
    df = df.rename(columns=cols)
    keep = ["date"] + [c for c in cols.values() if c in df]
    df = df[keep].copy()
    df["wx_station_id"] = station_id
    return df


def main():
    bikes = get_bike_stations()

    # 🔥 FIX: Nearest bike station calculations injected back here
    bikes = find_nearest_bike_station(bikes)

    wx = get_weather_stations()
    mapping = match_stations(bikes, wx)
    print(f"{len(mapping)} bike stations matched to "
          f"{mapping['wx_station_id'].nunique()} weather stations")

    weather = []
    for sid in mapping["wx_station_id"].unique():
        print(f"Downloading weather station {sid} ...")
        w = download_daily(sid)
        if not w.empty:
            weather.append(w)
    weather = pd.concat(weather, ignore_index=True)

    merged = mapping.merge(weather, on="wx_station_id", how="left")
    merged.to_csv(OUT_CSV, index=False)
    with pd.ExcelWriter(OUT_XLSX) as xl:
        merged.head(1_000_000).to_excel(xl, sheet_name="data", index=False)
        mapping.to_excel(xl, sheet_name="station_mapping", index=False)
    print(f"Saved {OUT_XLSX} and {OUT_CSV}")
    return merged


if __name__ == "__main__":
    df = main()


1072 bike stations matched to 5 weather stations
Saved bike_stations_weather.xlsx and bike_stations_weather.csv
